# EternityQuant — Colab ML 训练

在 Colab 的免费 T4 GPU 上训练 EternityQuant 的量化选股模型。

## 支持算法
- **LightGBM** (CPU) — 基线模型，qlib Alpha158 特征
- **MLP** (CUDA) — 自写全连接网络，158→512→256→128→1
- **GRU** (CUDA) — 自写时序模型，6×26 时序重塑，量化选股最佳
- **LSTM** (CUDA) — 与 GRU 同架构，3 门替代 2 门

## 输出
- 训练好的模型文件（`.pkl`）→ 下载回本地，用 `eq ml register/activate` 导入
- 训练指标（IC、ICIR、Rank IC）

---

## 1. 环境准备

安装依赖 + 克隆 EternityQuant 代码。

In [ ]:
# @title 安装依赖
import sys, os, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

# 基础依赖
_pip("numpy", "pandas", "typer", "python-dotenv")

# qlib（Colab 编译时间约 2 分钟）
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pyqlib>=0.9", "--no-build-isolation"])

# LightGBM
_pip("lightgbm>=4.0")

# PyTorch (Colab 预装 CUDA 版，只需确认)
import torch
print(f"PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\n依赖安装完成 ✓")

In [ ]:
# @title 克隆 EternityQuant
import os
REPO_URL = "https://github.com/your-username/EternityQuant.git"  # TODO: 替换为你的仓库地址
PROJECT_DIR = "/content/EternityQuant"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . --quiet
print(f"工作目录: {os.getcwd()}")

## 2. 准备 qlib 数据

### 方案 A：从云存储下载预打包数据（推荐，快）
如果有预打包的 qlib 数据（`.qlib_data/cn_data.tar.gz`），上传到 Colab 或从 Google Drive 挂载：

### 方案 B：用 baostock 在线拉取（慢，但无需提前准备）
Colab 网络可以访问 baostock（TCP 协议），直接运行 `eq ml update-data` 拉取 CSI300 数据。
约 30-60 分钟，取决于网络。

In [ ]:
# @title (可选) 挂载 Google Drive — 用于持久化数据和模型
from google.colab import drive
drive.mount("/content/drive")

# 创建符号链接，使得 qlib 数据可以直接从 Drive 读取
QLIB_DATA_DIR = "/content/.qlib_data/cn_data"
DRIVE_QLIB = "/content/drive/MyDrive/qlib_data/cn_data"

if os.path.exists(DRIVE_QLIB):
    !mkdir -p /content/.qlib_data
    !ln -sfn "{DRIVE_QLIB}" "{QLIB_DATA_DIR}"
    print(f"已链接 Drive 中的 qlib 数据: {DRIVE_QLIB}")
else:
    print(f"Drive 中未找到 qlib 数据，将在线拉取")

In [ ]:
# @title (方案 B) 在线拉取 qlib 数据
import os
QLIB_DATA_DIR = "/content/.qlib_data/cn_data"
if not os.path.exists(QLIB_DATA_DIR):
    !mkdir -p /content/.qlib_data
    # 先用 baostock 拉沪深 300 成分股数据（约 30-60 分钟）
    !cd /content/EternityQuant && python -c "
from eq.strategy.factors.ml_data_updater import update_qlib_data
result = update_qlib_data(start='2015-01-01', universe='csi300', workers=8, verbose=True)
print(f'数据更新完成: {result}')
"
    # 复制到 Drive 持久化
    !cp -r /content/.qlib_data/cn_data /content/drive/MyDrive/qlib_data/ 2>/dev/null || true
else:
    print(f"qlib 数据已存在: {QLIB_DATA_DIR}")

# 验证
!ls -la /content/.qlib_data/cn_data/calendars/ 2>/dev/null || echo "日历文件未找到"

## 3. 训练模型

### 3.1 LightGBM (CPU 基线)

In [ ]:
# @title 训练 LightGBM
import sys, os
os.chdir("/content/EternityQuant")
sys.path.insert(0, ".")

from eq.strategy.factors.ml_workflow import train

result = train(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="lightgbm",
    device="cpu",
    name="colab_lgbm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.2 MLP (CUDA)

In [ ]:
# @title 训练 MLP
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="mlp",
    device="cuda",
    name="colab_mlp_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.3 GRU (CUDA) — 推荐，量化选股最佳

In [ ]:
# @title 训练 GRU
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=4000,
    name="colab_gru_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  Epochs: {result['metrics']['epochs']}")
print(f"  模型文件: {result['model_path']}")

### 3.4 LSTM (CUDA)

In [ ]:
# @title 训练 LSTM
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="lstm",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=4000,
    name="colab_lstm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.5 超参搜索（自动找最佳 LSTM/GRU 参数）

In [ ]:
# @title GRU 超参网格搜索
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import search_lstm

results = search_lstm(
    universe="csi300",
    horizon=5,
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    device="cuda",
    fast=True,        # 快速模式：每组合 max_steps=50
    auto_train=True,  # 搜索完自动用最佳参数全量训练
    algo="gru",      # 可选 "lstm" 或 "gru"
)

# 打印 Top3 结果
print(f"\n搜索完成 {len(results)} 组合")
for i, r in enumerate(results[:3]):
    print(f"  #{i+1}: hidden={r['hidden_size']} layers={r['num_layers']} "
          f"lr={r['lr']} batch={r['batch_size']}  IC={r['ic']:+.4f}")

## 4. 导出模型回本地

训练好的模型文件在 `/content/EternityQuant/.eternityquant/ml_models/` 目录下。
将其下载到本地，然后用 `eq ml register` 和 `eq ml activate` 导入。

In [ ]:
# @title 列出所有训练好的模型
import os
models_dir = "/content/EternityQuant/.eternityquant/ml_models"
if os.path.exists(models_dir):
    for f in sorted(os.listdir(models_dir)):
        size = os.path.getsize(os.path.join(models_dir, f))
        print(f"  {f:45s}  {size/1024:.1f} KB")
else:
    print("未找到模型文件")

In [ ]:
# @title 打包下载所有模型到本地
import os
models_dir = "/content/EternityQuant/.eternityquant/ml_models"
if os.path.exists(models_dir) and os.listdir(models_dir):
    !tar czf /content/eternityquant_models.tar.gz -C {models_dir} .
    from google.colab import files
    files.download("/content/eternityquant_models.tar.gz")
else:
    print("没有模型文件可下载")

## 5. 回本地导入模型

```bash
# 解压模型文件
tar xzf eternityquant_models.tar.gz -C ~/.eternityquant/ml_models/

# 登记模型
eq ml register --name "colab_gru_csi300_h5" --universe csi300 \
    --features "Alpha158" --algo gru --horizon 5 \
    --train-period "2015-01-01~2020-08-31" \
    --model-path ~/.eternityquant/ml_models/torch_gru_csi300_5d.pkl \
    --metrics '{"ic": 0.15}' \
    --notes "Colab T4 GPU 训练"

# 激活模型
eq ml activate <model_id>

# 批量预测今日 Top 10
eq ml predict-batch <model_id> --top 10
```

---

## 附：Colab GPU 配置信息

In [ ]:
# @title GPU 信息
import torch, psutil, platform
print(f"系统: {platform.platform()}")
print(f"CPU: {psutil.cpu_count()} 核")
print(f"内存: {psutil.virtual_memory().total / 1e9:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU: 不可用（将使用 CPU）")